# Preparing a dataset

scPertEval reads a **preprocessed AnnData (`.h5ad`)** and recomputes everything else
(differential expression, controls, baselines) in memory. This tutorial shows how to turn a
*raw* perturb-seq dataset into that scPertEval-ready form, using three real datasets that each
illustrate a different wrinkle:

1. **`adamson16`** — the full walkthrough: cleaning messy perturbation labels and identifying controls.
2. **`wessels23`** — combination perturbations (`GENE1+GENE2`).
3. **`replogle22k562`** — the easy case (already-clean labels), where prep is essentially just normalize + trim.

The three sections are independent — run only the one you care about.

:::{admonition} This notebook is not run in CI
:class: warning
Each section downloads a real dataset from [scPerturb](http://projects.sanderlab.org/scperturb/)
(hundreds of MB to ~1.5 GB), so — unlike the other tutorials — it is **excluded from the
notebook CI job** and executed by hand. The code is copy-paste-able; point `DATA_DIR` at wherever
you keep your downloads.
:::

## What scPertEval needs

A scPertEval-ready file is an ordinary AnnData with just three things:

| Where | What | Notes |
|---|---|---|
| `adata.X` | **log-normalized** expression, cells × genes | `normalize_total(target_sum=1e4)` then `log1p`; sparse `float32` is ideal |
| `adata.obs["perturbation"]` | one **perturbation label per cell** | control cells share a single label, `"control"` |
| `adata.var_names` | **gene names** | the `var` index; used to report per-gene DE |

That's the whole contract. The `obs` column name and the control label are configurable
(`--perturbation-key` / `--control-label`), but `"perturbation"` and `"control"` are the
defaults and what we use here. Everything else in a typical raw file — extra layers, embeddings
in `obsm`, precomputed results in `uns`, dozens of QC columns — is **unused** and should be
dropped (see [Trimming](#trimming-and-why-it-matters) below). For the exact format reference see
the [Datasets](../user-guide/datasets.md) page.

In [1]:
from pathlib import Path
import urllib.request

import numpy as np
import scipy.sparse as sp
import scanpy as sc
import anndata as ad

# Where downloads are cached. Point this anywhere with a few GB free.
DATA_DIR = Path.home() / ".cache" / "scperteval_tutorial"
WORK_DIR = Path("prepared")  # cleaned outputs are written here
DATA_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# scPerturb harmonized RNA files live on Zenodo (record 13350497).
_ZENODO = "https://zenodo.org/records/13350497/files/{fname}?download=1"


def fetch(fname: str) -> Path:
    """Download an scPerturb file to DATA_DIR (cached; re-runs skip the download)."""
    dest = DATA_DIR / fname
    if dest.exists():
        print(f"cached: {fname}  ({dest.stat().st_size / 1e6:.0f} MB)")
        return dest
    print(f"downloading {fname} from scPerturb …")
    urllib.request.urlretrieve(_ZENODO.format(fname=fname), dest)
    print(f"saved:  {fname}  ({dest.stat().st_size / 1e6:.0f} MB)")
    return dest

### The recipe, in two functions

These two helpers are all the prep logic. They distill the parts of the benchmark's
preprocessing pipeline that actually matter to scPertEval — cleaning the perturbation column,
log-normalizing `X`, and stripping everything unused.

In [2]:
def clean_perturbation_labels(
    adata,
    source_col="perturbation",
    truncate_sep=None,
    control_pattern=None,
    control_values=("control", "ctrl", "Control", "CTRL", "Ctrl"),
    combo=False,
):
    """Build a clean, categorical ``obs['perturbation']`` and drop unlabeled cells.

    Steps (each optional, applied in order):
      - copy ``source_col`` to ``obs['perturbation']`` as strings
      - ``truncate_sep``: keep only the prefix before the separator (e.g. ``STAT1_pDS031`` -> ``STAT1``)
      - ``control_pattern``: regex; matching labels become ``"control"``
      - ``control_values``: exact labels that mean control become ``"control"``
      - ``combo``: join combination guides with ``+`` (``GENE1_GENE2`` -> ``GENE1+GENE2``)
    Cells whose label is missing (``nan``/empty) are removed.
    """
    lab = adata.obs[source_col].astype(str)
    if truncate_sep is not None:
        lab = lab.str.split(truncate_sep).str[0]
    if control_pattern is not None:
        lab = lab.mask(lab.str.contains(control_pattern, regex=True), "control")
    lab = lab.mask(lab.isin(list(control_values)), "control")
    if combo:
        lab = lab.str.replace("_", "+", regex=False)
    adata.obs["perturbation"] = lab.astype("category")
    keep = ~adata.obs["perturbation"].astype(str).isin(["nan", "NaN", "None", ""])
    return adata[keep].copy()


def to_scperteval(adata, target_sum=1e4, min_genes=200, min_cells=3):
    """Log-normalize, apply light QC, and strip to a minimal scPertEval-ready AnnData.

    Keeps only ``X`` (log-normalized, sparse float32), ``obs['perturbation']`` and the gene
    names. Everything else (raw-count layers, ``obsm``/``varm``/``uns``, extra ``obs`` columns)
    is dropped.
    """
    sc.pp.filter_cells(adata, min_genes=min_genes)  # drop near-empty cells
    sc.pp.filter_genes(adata, min_cells=min_cells)  # drop genes seen in <min_cells cells
    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)
    X = sp.csr_matrix(adata.X).astype(np.float32)
    out = ad.AnnData(X=X, obs=adata.obs[["perturbation"]].copy())
    out.var_names = adata.var_names
    return out

## 1 · adamson16 — cleaning labels and controls

The Adamson 2016 Perturb-seq file (K562, unfolded-protein-response screen) is a good worked
example because its raw perturbation labels need two kinds of cleanup:

- labels carry a **guide/plasmid suffix** — `SEC61A1_pDS031`, `OST4_pDS353` — so we split on `_`
  and keep the gene;
- non-targeting controls are encoded as parenthesized constructs — `63(mod)_pBA580` — which we
  catch with the regex `\(`.

`X` is raw UMI counts, so it also needs normalizing.

In [3]:
adamson_raw = sc.read_h5ad(fetch("AdamsonWeissman2016_GSM2406681_10X010.h5ad"))
print(adamson_raw)
print("\nX is raw counts:", adamson_raw.X.dtype, "| max:", adamson_raw.X[:500].toarray().max())
print("example raw labels:", list(adamson_raw.obs["perturbation"].astype(str).unique()[:6]))

cached: AdamsonWeissman2016_GSM2406681_10X010.h5ad  (471 MB)


AnnData object with n_obs × n_vars = 65337 × 32738
    obs: 'perturbation', 'read count', 'UMI count', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'ncounts', 'ngenes', 'percent_mito', 'percent_ribo', 'nperts'
    var: 'ensembl_id', 'ncounts', 'ncells'

X is raw counts: float32 | max: 864.0
example raw labels: ['63(mod)_pBA580', 'OST4_pDS353', 'SEC61A1_pDS031', 'EIF2B4_pDS491', 'SRPR_pDS482', 'IER3IP1_pDS002']


Split off the plasmid suffix (`truncate_sep="_"`) and route the `(mod)` controls to `"control"`
(`control_pattern=r"\("`):

In [4]:
adamson = clean_perturbation_labels(adamson_raw, truncate_sep="_", control_pattern=r"\(")
vc = adamson.obs["perturbation"].value_counts()
print("perturbations (incl. control):", adamson.obs["perturbation"].nunique())
print("control cells:", int(vc.get("control", 0)))
print("example cleaned labels:", list(vc.index[:8]))

perturbations (incl. control): 92
control cells: 7295
example cleaned labels: ['control', 'IER3IP1', 'ASCC3', 'SEC61B', 'DNAJC19', 'YIPF5', 'SCYL1', 'SEC61A1']


Now log-normalize and strip to the minimal form:

In [5]:
adamson_ready = to_scperteval(adamson)
print(adamson_ready)
x0 = adamson_ready.X[:500].toarray()
print(
    "\nX is log-normalized now:",
    adamson_ready.X.dtype,
    "| max:",
    round(float(x0.max()), 3),
    "| integer-valued:",
    np.allclose(x0, np.round(x0)),
)

AnnData object with n_obs × n_vars = 62724 × 20544
    obs: 'perturbation'

X is log-normalized now: float32 | max: 6.593 | integer-valued: False


### Trimming, and why it matters

Public perturb-seq files are usually bloated for this use case. The most common culprit is a
**redundant raw-count layer** kept alongside the normalized matrix — exactly what the benchmark
pipeline drops with `include_raw_counts: false`. scPertEval only needs the normalized `X`, so
carrying the counts (plus every QC column and embedding) just makes the file bigger and slower to
load. The comparison below writes a "naive" export that keeps the raw counts and all metadata,
next to our minimal file:

In [ ]:
# minimal: exactly what scPertEval reads
minimal_path = WORK_DIR / "adamson16.h5ad"
adamson_ready.write_h5ad(minimal_path, compression="gzip")

# naive: normalized X + a redundant raw-count layer + all obs/var metadata
naive = adamson_ready.copy()
counts = adamson_raw[adamson_ready.obs_names, adamson_ready.var_names].X
naive.layers["counts"] = sp.csr_matrix(counts).astype(np.float32)
naive.obs = adamson_raw.obs.loc[adamson_ready.obs_names].copy()
naive_path = WORK_DIR / "adamson16_naive.h5ad"
naive.write_h5ad(naive_path, compression="gzip")


def mb(p):
    """File size in megabytes."""
    return p.stat().st_size / 1e6


print(f"minimal (X + perturbation)         : {mb(minimal_path):5.0f} MB")
print(f"naive   (+ raw counts + metadata)  : {mb(naive_path):5.0f} MB")
print(f"raw download                       : {mb(DATA_DIR / 'AdamsonWeissman2016_GSM2406681_10X010.h5ad'):5.0f} MB")
naive_path.unlink()  # tidy up the illustration

### Confirm it works

Point `calibrate` at the file we just wrote. It runs, so the file is in the right shape:

In [7]:
from scperteval.cli import main

main(["calibrate", str(minimal_path), "-p", "pearson_ctrl,mse", "--subsample", "2000", "--out-dir", str(WORK_DIR)])


adamson16 · t-test · subsample=2000 · seed=42 · calibrator=drf

protocol                   representation space          mean    median
-----------------------------------------------------------------------
pearson_ctrl               centroid       full          0.433     0.467
mse                        centroid       full          0.416     0.404

-> prepared/adamson16__2026-07-08T120602__drf.csv


## 2 · wessels23 — combination perturbations

Wessels 2023 is a **combinatorial** CRISPRi screen: most cells received *two* guides. In the raw
labels the two genes are joined with `_` (`IKZF1_SMARCD1`, `DOT1L_INTS1`), while single-gene cells
carry a bare gene name (`INTS1`) and controls are already labelled `control`.

Passing `combo=True` rewrites the joiner to `+`, giving canonical combo labels like
`IKZF1+SMARCD1`. Both single- and double-gene conditions are kept — nothing is excluded here;
which of them a study treats as *prediction targets* is a downstream modelling choice (for the
hosted `wessels23`, combos are the targets and singles are additive-model building blocks; see
[Datasets](../user-guide/datasets.md)).

In [ ]:
wessels_raw = sc.read_h5ad(fetch("WesselsSatija2023.h5ad"))
print("raw obsm (dropped on trim):", list(wessels_raw.obsm.keys()))
print("example raw labels:", list(wessels_raw.obs["perturbation"].astype(str).unique()[:6]))

wessels = clean_perturbation_labels(wessels_raw, combo=True)
wessels_ready = to_scperteval(wessels)

labels = wessels_ready.obs["perturbation"].astype(str)
combos = sorted({lbl for lbl in labels.unique() if "+" in lbl})
singles = sorted({lbl for lbl in labels.unique() if "+" not in lbl and lbl != "control"})
print(
    f"\n{wessels_ready.shape[0]} cells | {len(combos)} combo + {len(singles)} single "
    f"perturbations + control ({int((labels == 'control').sum())} cells)"
)
print("example combos:", combos[:5])

wessels_path = WORK_DIR / "wessels23.h5ad"
wessels_ready.write_h5ad(wessels_path, compression="gzip")
print("wrote", wessels_path, f"({wessels_path.stat().st_size / 1e6:.0f} MB)")

## 3 · replogle22k562 — the easy case

Replogle 2022 (K562 essential-gene screen) is the kind of file you'll usually have: perturbation
labels are already clean gene symbols (`BUB1`, `RPL3`), controls are labelled `control`, and there
are no combos. Prep is just normalize + trim. It is also the largest of the three (~1.5 GB, ~310k
cells), so the download and load take a while.

In [9]:
replogle_raw = sc.read_h5ad(fetch("ReplogleWeissman2022_K562_essential.h5ad"))
print(replogle_raw.shape, "| example labels:", list(replogle_raw.obs["perturbation"].astype(str).unique()[:6]))

# no truncation, no regex — the defaults already map "control" and clean symbols pass through
replogle = clean_perturbation_labels(replogle_raw)
replogle_ready = to_scperteval(replogle)

replogle_path = WORK_DIR / "replogle22k562.h5ad"
replogle_ready.write_h5ad(replogle_path, compression="gzip")
print(
    replogle_ready.shape,
    f"| {replogle_ready.obs['perturbation'].nunique()} perturbations",
    f"| {replogle_path.stat().st_size / 1e6:.0f} MB",
)

cached: ReplogleWeissman2022_K562_essential.h5ad  (1547 MB)


(310385, 8563) | example labels: ['NAF1', 'BUB1', 'UBL5', 'C9orf16', 'TIMM9', 'SMG5']


(310385, 8563) | 2058 perturbations | 2450 MB


## Recap

The recipe is the same every time:

1. **Clean `obs["perturbation"]`** — one label per cell, all controls collapsed to `"control"`,
   guide suffixes stripped, combos joined with `+`, unlabeled cells dropped.
2. **Log-normalize `X`** — `normalize_total(1e4)` + `log1p`, with light QC filtering.
3. **Trim** — keep only `X`, `obs["perturbation"]` and the gene names; write sparse `float32`,
   gzip-compressed.

The per-dataset differences are entirely in step 1 (which is why `clean_perturbation_labels`
takes a few flags). Once written, pass the file straight to `scperteval calibrate` /
`scperteval score` — see the [Datasets](../user-guide/datasets.md) page for the format reference
and the ready-made hosted datasets.